# Parameter Bank Shapes and Sharding Analysis

This notebook analyzes the shapes and sharding structures of the `attn_bank` and `mlp_bank` from the Muon optimizer implementation in `train_gpt_muon.py`.

## Network Architecture Parameters
The parameter banks apply to the dense layers given the model configurations:
- **Attention Layers (`num_attn_layers`)**: `10`
- **MLP Layers**: `12`
- **Model Dimension (`model_dim`, `hdim`)**: `768`
- **MLP Hidden Dimension (`mlp_hdim`)**: `3072` (or effectively `4 * 768`)

In [1]:
import torch
import torch.nn as nn

num_attn_layers = 10
model_dim = hdim = 768
mlp_hdim = 3072

# Attention Bank
attn_bank = nn.Parameter(torch.empty(num_attn_layers, 4 * model_dim, hdim))
attn_bank_reshape = (num_attn_layers * 4, hdim, hdim)

print("attn_bank shape:")
print(f"  Original memory shape: {attn_bank.shape}")
print(f"  Reshaped for sharding: {torch.Size(attn_bank_reshape)}\n\n")

# MLP Bank
mlp_bank = nn.Parameter(torch.empty(12, 2, mlp_hdim, model_dim))
mlp_bank_reshape = (24, mlp_hdim, model_dim)

print("mlp_bank shape:")
print(f"  Original memory shape: {mlp_bank.shape}")
print(f"  Reshaped for sharding: {torch.Size(mlp_bank_reshape)}")


attn_bank shape:
  Original memory shape: torch.Size([10, 3072, 768])
  Reshaped for sharding: torch.Size([40, 768, 768])


mlp_bank shape:
  Original memory shape: torch.Size([12, 2, 3072, 768])
  Reshaped for sharding: torch.Size([24, 3072, 768])


### Sharding Properties
The reshapes defined above guarantee that both parameter banks are structurally divisible across the standard model configurations without incurring ragged partitions:

- `attn_bank` flattens its leading dimension out to `40`, allowing perfect division across `world_size=8` GPUs (yielding `40 / 8 = 5` attention matrices per worker for orthogonalization).
- `mlp_bank` flattens its leading dimension out to `24`, allowing perfect division across `world_size=8` GPUs (yielding `24 / 8 = 3` projection matrices per worker).